## Bronze to Silver


#### Clean, transform, and standardize Bronze data into a structured Silver Delta table

Run the Configurations file

In [0]:
%run "/Workspace/bank-project/00-Configurations"

Import necessary functions, libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging

Create table pipeline_logs

In [0]:
# %sql
# USE CATALOG cc_trans_catalog;
# CREATE SCHEMA IF NOT EXISTS cc_trans_catalog.audit;
# CREATE TABLE IF NOT EXISTS cc_trans_catalog.audit.pipeline_logs (
#     pipeline_name STRING,
#     step_name STRING,
#     status STRING,
#     record_count BIGINT,
#     error_message STRING,
#     log_time TIMESTAMP
# )
# USING DELTA;


Silver transformation logic using OOPs

In [0]:
class BronzeToSilver:

    def __init__(self, spark, bronze_table, silver_table, silver_base_path):
        self.spark = spark
        self.bronze_table = bronze_table
        self.silver_table = silver_table
        self.silver_base_path = silver_base_path

        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger("BronzeToSilverPipeline")

        self.pipeline_name = "bronze_to_silver_pipeline"

    def read_bronze(self):
        self.logger.info("Reading bronze table")
        df = self.spark.read.table(self.bronze_table)
        return df

    def convert_datatypes(self, df):
        self.logger.info("Converting datatypes of columns.")
        
        df = df.withColumn(
                "trans_date_trans_time",
                to_timestamp(substring(col("trans_date_trans_time"), 1, 19), "yyyy-MM-dd HH:mm:ss")
            ) \
            .withColumn("amt", col("amt").cast("double")) \
            .withColumn("zip", col("zip").cast("integer")) \
            .withColumn("lat", col("lat").cast("double")) \
            .withColumn("long", col("long").cast("double")) \
            .withColumn("city_pop", col("city_pop").cast("integer")) \
            .withColumn("dob", to_date(substring(col("dob"), 1, 10), "yyyy-MM-dd")) \
            .withColumn("unix_time", col("unix_time").cast("long")) \
            .withColumn("merch_lat", col("merch_lat").cast("double")) \
            .withColumn("merch_long", col("merch_long").cast("double")) \
            .withColumn("is_fraud", col("is_fraud").cast("integer")) \
            .withColumn("merch_zipcode", col("merch_zipcode").cast("integer"))
        return df

    def handle_nulls(self, df):
        self.logger.info("Handling null values")
        df = df.fillna({"merch_zipcode": 99999})
        return df

    def rename_columns(self, df):
        self.logger.info("Renaming columns")
        df = df.withColumnRenamed("cc_num", "card_number") \
            .withColumnRenamed("trans_date_trans_time", "transaction_time") \
            .withColumnRenamed("amt", "transaction_amount") \
            .withColumnRenamed("first","first_name")\
            .withColumnRenamed("last","last_name")\
            .withColumnRenamed("city_pop", "city_population") \
            .withColumnRenamed("zip", "zipcode") \
            .withColumnRenamed("lat", "latitude") \
            .withColumnRenamed("long", "longitude")\
            .withColumnRenamed("trans_num", "transaction_number")\
            .withColumnRenamed("unix_time", "unix_timestamp")\
            .withColumnRenamed("merch_lat", "merchant_latitude")\
            .withColumnRenamed("merch_long", "merchant_longitude")\
            .withColumnRenamed("merch_zipcode", "merchant_zipcode")
        return df
    
    def add_ingestion_time(self, df):
        self.logger.info("Adding ingestion timestamp")
        return df.withColumn("ingestion_time", current_timestamp())

    def select_columns(self, df):
        self.logger.info("Selecting the required columns")
        df = df.select(
            "transaction_number",
            "transaction_time",
            "card_number",
            "merchant",
            "category",
            "transaction_amount",
            "first_name",
            "last_name",
            "gender",
            "street",
            "city",
            "state",
            "zipcode",
            "latitude",
            "longitude",
            "city_population",
            "job",
            "dob",
            "merchant_latitude",
            "merchant_longitude",
            "merchant_zipcode",
            "ingestion_time"
        )
        return df

    def write_silver(self, df):
        self.logger.info("Writing silver table")
        path = f"{self.silver_base_path}/credit_card_transactions"

        print("Writing to path:", path)

        df.write.format("delta") \
            .mode("overwrite") \
            .option("path", path) \
            .saveAsTable(self.silver_table)
    
    def log_to_table(self, step, status, record_count=0, error_msg=None):

        if error_msg is None:
            error_msg = ""

        schema = StructType([
            StructField("pipeline_name", StringType(), True),
            StructField("step_name", StringType(), True),
            StructField("status", StringType(), True),
            StructField("record_count", LongType(), True),  
            StructField("error_message", StringType(), True)
        ])

        log_data = [(
            self.pipeline_name,
            step,
            status,
            record_count,
            error_msg
        )]

        log_df = self.spark.createDataFrame(log_data, schema) \
            .withColumn("log_time", current_timestamp())

        log_df.write.mode("append") \
            .saveAsTable("cc_trans_catalog.audit.pipeline_logs")

        self.logger.info(f"Logged step: {step}, status: {status}")
    

    def run_pipeline(self):
        try:
            df = self.read_bronze()
            self.log_to_table("read_bronze", "SUCCESS", df.count())

            df = self.convert_datatypes(df)
            self.log_to_table("convert_datatypes", "SUCCESS")

            df = self.handle_nulls(df)
            self.log_to_table("handle_nulls", "SUCCESS")

            df = self.rename_columns(df)
            self.log_to_table("rename_columns", "SUCCESS")

            df = self.add_ingestion_time(df)
            self.log_to_table("add_ingestion_time", "SUCCESS")

            df = self.select_columns(df)
            record_count = df.count()
            self.log_to_table("select_columns", "SUCCESS", record_count)

            self.write_silver(df)
            self.log_to_table("write_silver", "SUCCESS", record_count)

            return df

        except Exception as e:
            self.log_to_table("pipeline_failed", "FAILED", 0, str(e))
            self.logger.error(str(e))
            raise

Object creation

In [0]:
pipeline = BronzeToSilver(
    spark=spark,
    bronze_table="cc_trans_catalog.trans_bronze.cc_transactions_raw",
    silver_table="cc_trans_catalog.trans_silver.cc_transactions_silver",
    silver_base_path=silver_path
)


Execute pipeline

In [0]:
pipeline.run_pipeline()

INFO:BronzeToSilverPipeline:Reading bronze table
INFO:BronzeToSilverPipeline:Logged step: read_bronze, status: SUCCESS
INFO:BronzeToSilverPipeline:Converting datatypes of columns.
INFO:BronzeToSilverPipeline:Logged step: convert_datatypes, status: SUCCESS
INFO:BronzeToSilverPipeline:Handling null values
INFO:BronzeToSilverPipeline:Logged step: handle_nulls, status: SUCCESS
INFO:BronzeToSilverPipeline:Renaming columns
INFO:BronzeToSilverPipeline:Logged step: rename_columns, status: SUCCESS
INFO:BronzeToSilverPipeline:Adding ingestion timestamp
INFO:BronzeToSilverPipeline:Logged step: add_ingestion_time, status: SUCCESS
INFO:BronzeToSilverPipeline:Selecting the required columns
INFO:BronzeToSilverPipeline:Logged step: select_columns, status: SUCCESS
INFO:BronzeToSilverPipeline:Writing silver table


Writing to path: abfss://transaction-data@rcprstrgacc.dfs.core.windows.net/silver/credit_card_transactions


INFO:BronzeToSilverPipeline:Logged step: write_silver, status: SUCCESS


DataFrame[transaction_number: string, transaction_time: timestamp, card_number: string, merchant: string, category: string, transaction_amount: double, first_name: string, last_name: string, gender: string, street: string, city: string, state: string, zipcode: int, latitude: double, longitude: double, city_population: int, job: string, dob: date, merchant_latitude: double, merchant_longitude: double, merchant_zipcode: int, ingestion_time: timestamp]